# Product Price Prediction (Amazon ML Challenge 2025)

**Problem:** predict the price of a product from its catalog text (title, bullet points, pack size) and its product image.
**Metric:** SMAPE (Symmetric Mean Absolute Percentage Error), where lower is better. Its range is 0 to 200%.

**Pipeline**
1. Data validation and cleaning
2. Feature engineering from raw listings (quantity, unit, pack size, brand, text stats)
3. Text embeddings (TF-IDF + sentence-transformer)
4. Image embeddings (CLIP)
5. LightGBM models compared step by step with 5-fold cross-validation, then an ensemble

**Dataset:** `train.csv` (75,000 products) was shared by Amazon with participants of the Amazon ML Challenge 2025. I uploaded it manually to this repo as `dataset/train.csv.gz`.

**How to run on Kaggle:** turn on GPU and Internet in the right panel, then Run All. The notebook downloads the dataset from this repo and the product images itself.

In [ ]:
import os, re, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.linear_model import Ridge
warnings.filterwarnings('ignore'); sns.set_theme(style='whitegrid')

SEED = 42
N_PRODUCTS = 12_000          # products used for training and cross-validation
N_FOLDS = 5
USE_TEXT_EMB = True          # sentence-transformer embeddings (needs internet once)
USE_IMAGES = True            # CLIP image embeddings (needs internet + GPU recommended)
np.random.seed(SEED)

def smape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return 100 * np.mean(2 * np.abs(p - y) / (np.abs(y) + np.abs(p) + 1e-9))

## 1. Load data

In [ ]:
# The dataset was shared by Amazon with challenge participants; I uploaded it to this repo (dataset/train.csv.gz)
CANDIDATES = ['../dataset/train.csv.gz', 'dataset/train.csv.gz', '../dataset/train.csv', 'dataset/train.csv']
path = next((p for p in CANDIDATES if os.path.exists(p)), None)
if path is None:   # e.g. on Kaggle: download it from this GitHub repo
    path = 'https://raw.githubusercontent.com/GAURAV-AgarwL/product-price-prediction/main/dataset/train.csv.gz'
full = pd.read_csv(path)
print('Full training file:', full.shape)
full.head(3)

## 2. Data validation

In [ ]:
print('Missing values:'); print(full.isna().sum(), '\n')
print('Duplicate sample_id :', full['sample_id'].duplicated().sum())
print('Duplicate catalog text:', full['catalog_content'].duplicated().sum())
print('Prices <= 0          :', (full['price'] <= 0).sum())
print(full['price'].describe().round(2))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
full['price'].clip(upper=200).hist(bins=60, ax=ax[0]); ax[0].set_title('Price (clipped at $200)')
np.log1p(full['price']).hist(bins=60, ax=ax[1]); ax[1].set_title('log(1 + price)')
plt.tight_layout(); plt.show()

Prices are heavily right-skewed (median about $14, max $2,796), so I train on **log(1 + price)** and convert predictions back.
Below I keep valid rows, drop exact duplicate listings, and take a random sample of products for the experiments.

In [ ]:
clean = full.dropna(subset=['catalog_content', 'price'])
clean = clean[clean['price'] > 0].drop_duplicates('catalog_content')
df = clean.sample(N_PRODUCTS, random_state=SEED).reset_index(drop=True)
print(f'Rows after validation: {len(clean):,}   Products used: {len(df):,}')

## 3. Feature engineering from raw listings

In [ ]:
UNIT_MAP = {'ounce': 'oz', 'ounces': 'oz', 'oz': 'oz', 'fl oz': 'floz', 'fluid ounce': 'floz', 'fl. oz': 'floz',
            'fluid ounces': 'floz', 'count': 'count', 'ct': 'count', 'pound': 'lb', 'pounds': 'lb', 'lb': 'lb',
            'gram': 'g', 'grams': 'g', 'g': 'g', 'kilogram': 'kg', 'kg': 'kg', 'milliliter': 'ml', 'ml': 'ml',
            'liter': 'l', 'litre': 'l', 'l': 'l', 'each': 'count', 'piece': 'count', 'pack': 'count'}
TO_OZ = {'oz': 1, 'floz': 1.043, 'lb': 16, 'g': 0.03527, 'kg': 35.27, 'ml': 0.0338, 'l': 33.81}

def parse(text):
    t = str(text)
    value = re.search(r'Value:\s*([\d.]+)', t)
    unit = re.search(r'Unit:\s*([^\n]+)', t)
    name = re.search(r'Item Name:\s*([^\n]+)', t)
    pack = (re.search(r'(?:pack|case|set|box) of\s*(\d+)', t, re.I)
            or re.search(r'(\d+)\s*(?:-|\s)?(?:count|ct|pack|pk)\b', t, re.I))
    u = UNIT_MAP.get(unit.group(1).strip().lower(), 'other') if unit else 'none'
    v = float(value.group(1)) if value else np.nan
    item = name.group(1) if name else ''
    return {
        'value': v, 'unit': u,
        'value_oz': v * TO_OZ[u] if u in TO_OZ and v == v else np.nan,
        'pack': float(pack.group(1)) if pack else 1.0,
        'brand': item.split()[0].lower().strip(',.') if item else 'none',
        'n_bullets': t.count('Bullet Point'),
        'text_len': len(t), 'name_len': len(item),
        'has_desc': int('Product Description' in t),
        'n_numbers': len(re.findall(r'\d+', item)),
        'premium_word': int(bool(re.search(r'organic|premium|gourmet|natural|artisan', t, re.I))),
    }

feat = pd.DataFrame([parse(t) for t in df['catalog_content']])
feat['value_per_pack'] = feat['value'] / feat['pack']
print('Rows with a quantity value:', feat['value'].notna().mean().round(3))
print(feat['unit'].value_counts().head(8))
feat.head()

## 4. Text embeddings

In [ ]:
text_emb = None
if USE_TEXT_EMB:
    try:
        from sentence_transformers import SentenceTransformer
        st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
        text_emb = st.encode(df['catalog_content'].str[:1000].tolist(), batch_size=128, show_progress_bar=True)
        print('Sentence embeddings:', text_emb.shape)
    except Exception as e:
        print('Sentence-transformer not available, continuing without it:', type(e).__name__)

## 5. Image embeddings

In [ ]:
img_emb = None
if USE_IMAGES:
    try:
        import torch, requests
        from io import BytesIO
        from PIL import Image
        from concurrent.futures import ThreadPoolExecutor
        from transformers import CLIPModel, CLIPProcessor

        def fetch(url):
            try:
                r = requests.get(url, timeout=10)
                return Image.open(BytesIO(r.content)).convert('RGB').resize((224, 224))
            except Exception:
                return None

        t0 = time.time()
        with ThreadPoolExecutor(32) as ex:
            images = list(ex.map(fetch, df['image_link']))
        ok = np.array([im is not None for im in images])
        print(f'Images downloaded: {ok.sum():,} of {len(ok):,} in {time.time() - t0:.0f}s')

        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        clip = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
        proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')
        img_emb = np.full((len(df), 512), np.nan, dtype=np.float32)
        idx = np.where(ok)[0]
        with torch.no_grad():
            for s in range(0, len(idx), 256):
                b = idx[s:s + 256]
                inp = proc(images=[images[i] for i in b], return_tensors='pt').to(device)
                # vision encoder + projection directly (works on old and new transformers versions)
                e = clip.visual_projection(clip.vision_model(pixel_values=inp['pixel_values']).pooler_output)
                img_emb[b] = torch.nn.functional.normalize(e, dim=-1).cpu().numpy()
        del images
        print('Image embeddings:', img_emb.shape)
    except Exception as e:
        print('Image step not available, continuing without images:', type(e).__name__, e)

## 6. Models, step by step (5-fold cross-validation)

Every number below is **out-of-fold SMAPE** on the same products, so each step is a fair comparison.
TF-IDF, SVD and PCA are fit **inside each fold** on training rows only, to avoid leakage.

In [ ]:
y = np.log1p(df['price'].values)
price = df['price'].values
kf = KFold(N_FOLDS, shuffle=True, random_state=SEED)
folds = list(kf.split(df))

tab_cols = ['value', 'value_oz', 'pack', 'value_per_pack', 'n_bullets', 'text_len', 'name_len',
            'has_desc', 'n_numbers', 'premium_word']

def fold_matrix(tr, va, use_tfidf=False, use_st=False, use_img=False):
    Xtr = feat.iloc[tr][tab_cols].reset_index(drop=True)
    Xva = feat.iloc[va][tab_cols].reset_index(drop=True)
    # categorical: unit, plus brand frequency computed on train rows only
    for X, rows in [(Xtr, tr), (Xva, va)]:
        X['unit'] = pd.Categorical(feat['unit'].iloc[rows].values, categories=sorted(feat['unit'].unique()))
    bfreq = feat['brand'].iloc[tr].value_counts()
    Xtr['brand_freq'] = feat['brand'].iloc[tr].map(bfreq).values
    Xva['brand_freq'] = feat['brand'].iloc[va].map(bfreq).fillna(0).values
    parts_tr, parts_va = [Xtr], [Xva]
    if use_tfidf:
        tf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50_000, sublinear_tf=True)
        Ttr = tf.fit_transform(df['catalog_content'].iloc[tr]); Tva = tf.transform(df['catalog_content'].iloc[va])
        svd = TruncatedSVD(100, random_state=SEED).fit(Ttr)
        parts_tr.append(pd.DataFrame(svd.transform(Ttr)).add_prefix('svd_'))
        parts_va.append(pd.DataFrame(svd.transform(Tva)).add_prefix('svd_'))
    if use_st and text_emb is not None:
        pca = PCA(64, random_state=SEED).fit(text_emb[tr])
        parts_tr.append(pd.DataFrame(pca.transform(text_emb[tr])).add_prefix('st_'))
        parts_va.append(pd.DataFrame(pca.transform(text_emb[va])).add_prefix('st_'))
    if use_img and img_emb is not None:
        fill = np.nan_to_num(img_emb, nan=0.0)
        pca = PCA(64, random_state=SEED).fit(fill[tr])
        parts_tr.append(pd.DataFrame(pca.transform(fill[tr])).add_prefix('img_'))
        parts_va.append(pd.DataFrame(pca.transform(fill[va])).add_prefix('img_'))
    return pd.concat(parts_tr, axis=1), pd.concat(parts_va, axis=1)

LGB_BASE = dict(n_estimators=600, learning_rate=0.05, num_leaves=31, random_state=SEED, verbose=-1)

def run_lgb(params, **flags):
    oof = np.zeros(len(df))
    for tr, va in folds:
        Xtr, Xva = fold_matrix(tr, va, **flags)
        m = lgb.LGBMRegressor(**params).fit(Xtr, y[tr])
        oof[va] = m.predict(Xva)
    return oof

results, oofs = {}, {}
def log_result(name, oof):
    oofs[name] = oof; results[name] = smape(price, np.expm1(oof))
    print(f'{name:45s} SMAPE = {results[name]:.2f}%')

In [ ]:
# Baseline: LightGBM on the raw listing fields only (quantity value + unit), no engineering
oof = np.zeros(len(df))
for tr, va in folds:
    Xb = feat[['value']].assign(unit=feat['unit'].astype('category'))
    m = lgb.LGBMRegressor(**LGB_BASE).fit(Xb.iloc[tr], y[tr]); oof[va] = m.predict(Xb.iloc[va])
log_result('1. Baseline (raw value + unit)', oof)

log_result('2. + engineered listing features', run_lgb(LGB_BASE))
log_result('3. + TF-IDF text (SVD)', run_lgb(LGB_BASE, use_tfidf=True))
if text_emb is not None:
    log_result('4. + sentence-transformer text embeddings', run_lgb(LGB_BASE, use_tfidf=True, use_st=True))
if img_emb is not None:
    log_result('5. + CLIP image embeddings', run_lgb(LGB_BASE, use_tfidf=True, use_st=True, use_img=True))

### Tuning LightGBM with K-fold cross-validation

In [ ]:
flags = dict(use_tfidf=True, use_st=text_emb is not None, use_img=img_emb is not None)
grid = [dict(num_leaves=nl, learning_rate=lr, n_estimators=ne, colsample_bytree=cs, min_child_samples=mc)
        for nl, lr, ne, cs, mc in [(31, 0.03, 1500, 0.5, 20), (63, 0.03, 1500, 0.5, 20),
                                   (63, 0.02, 2500, 0.4, 30), (127, 0.02, 2500, 0.4, 40)]]
tune = []
for g in grid:
    params = dict(LGB_BASE, **g)
    s = smape(price, np.expm1(run_lgb(params, **flags)))
    tune.append({**g, 'cv_smape': round(s, 2)}); print(g, '->', round(s, 2))
tune = pd.DataFrame(tune).sort_values('cv_smape')
best = dict(LGB_BASE, **tune.iloc[0].drop('cv_smape').to_dict())
for k in ['num_leaves', 'n_estimators', 'min_child_samples']: best[k] = int(best[k])
log_result('6. Tuned LightGBM (all features)', run_lgb(best, **flags))

### Ensemble

In [ ]:
# Model A: tuned LightGBM, averaged over 3 seeds
seed_oofs = [run_lgb(dict(best, random_state=s), **flags) for s in [42, 7, 2025]]
lgb_oof = np.mean(seed_oofs, axis=0)

# Model B: Ridge on full TF-IDF text (word + character n-grams)
ridge_oof = np.zeros(len(df))
for tr, va in folds:
    tw = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=80_000, sublinear_tf=True)
    tc = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, max_features=80_000, sublinear_tf=True)
    from scipy.sparse import hstack
    Xtr = hstack([tw.fit_transform(df['catalog_content'].iloc[tr]), tc.fit_transform(df['catalog_content'].iloc[tr])])
    Xva = hstack([tw.transform(df['catalog_content'].iloc[va]), tc.transform(df['catalog_content'].iloc[va])])
    ridge_oof[va] = Ridge(alpha=3.0).fit(Xtr, y[tr]).predict(Xva)
print(f'Ridge (TF-IDF) alone SMAPE = {smape(price, np.expm1(ridge_oof)):.2f}%')

# Blend weight chosen by nested CV: pick w on 4 folds, apply to the 5th
ens = np.zeros(len(df)); ws = np.linspace(0, 1, 21)
for tr, va in folds:
    w = min(ws, key=lambda w: smape(price[tr], np.expm1(w * lgb_oof[tr] + (1 - w) * ridge_oof[tr])))
    ens[va] = w * lgb_oof[va] + (1 - w) * ridge_oof[va]
log_result('7. Final ensemble (LightGBM x3 seeds + Ridge)', ens)

## 7. Results

In [ ]:
res = pd.Series(results).rename('CV SMAPE %').round(2).to_frame()
display(res)
base, final = results['1. Baseline (raw value + unit)'], results['7. Final ensemble (LightGBM x3 seeds + Ridge)']
print(f'SMAPE: {base:.2f}% -> {final:.2f}%  ({(base - final) / base:.1%} relative improvement)')

fig, ax = plt.subplots(figsize=(9, 4))
res['CV SMAPE %'].plot(kind='barh', ax=ax, color='#4C72B0'); ax.invert_yaxis()
ax.set_title('5-fold CV SMAPE at each step (lower is better)'); ax.set_xlabel('SMAPE %')
plt.tight_layout(); os.makedirs('figures', exist_ok=True); plt.savefig('figures/smape_steps.png'); plt.show()

In [ ]:
# Final model on all products: feature importance by group
Xall, _ = fold_matrix(np.arange(len(df)), np.arange(5), **flags)
final_model = lgb.LGBMRegressor(**best).fit(Xall, y)
imp = pd.Series(final_model.booster_.feature_importance('gain'), index=Xall.columns)
group = imp.groupby(lambda c: 'TF-IDF text' if c.startswith('svd_') else 'Sentence embedding' if c.startswith('st_')
                    else 'Image (CLIP)' if c.startswith('img_') else 'Listing features').sum()
display((group / group.sum() * 100).round(1).sort_values(ascending=False).rename('Gain share %').to_frame())
display(imp[~imp.index.str.match(r'(svd|st|img)_')].sort_values(ascending=False).head(10).round(0).to_frame('gain'))

In [ ]:
summary = {
    'products_used': len(df),
    'folds': N_FOLDS,
    'text_embeddings_used': text_emb is not None,
    'image_embeddings_used': img_emb is not None,
    'images_downloaded': int(np.isfinite(img_emb[:, 0]).sum()) if img_emb is not None else 0,
    'cv_smape_by_step': {k: round(v, 2) for k, v in results.items()},
    'baseline_smape': round(base, 2), 'final_smape': round(final, 2),
    'relative_improvement_pct': round((base - final) / base * 100, 1),
    'best_lgbm_params': {k: best[k] for k in ['num_leaves', 'learning_rate', 'n_estimators', 'colsample_bytree', 'min_child_samples']},
}
print(json.dumps(summary, indent=2))